In [1]:
# =========================
# STEP 4 — FEATURE ENGINEERING (single cell, independent)
# Input priority:
#   1) /user/data/clustering/panel_with_cluster_id
#   2) fallback: /user/data/preprocess/full_panel + cluster_assignments_final.csv
# Output:
#   - /user/data/feature_engineering/feature_panel_model_ready
#   - /workspace/code/kmeans/feature_meta/feature_null_summary.csv
#   - /workspace/code/kmeans/feature_meta/feature_metadata.json
# =========================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.storagelevel import StorageLevel

# -------------------------
# 1) CONFIG
# -------------------------
PANEL_WITH_CLUSTER_PATH = "/user/data/clustering/panel_with_cluster_id"
FULL_PANEL_PATH = "/user/data/preprocess/full_panel"
ASSIGNMENTS_CSV = "/workspace/code/kshape/clustering_meta/cluster_assignments_final.csv"

OUT_DIR = "/user/data/feature_engineering"
FEATURE_OUT_PATH = f"{OUT_DIR}/feature_panel_model_ready"

LOCAL_META_DIR = "/workspace/code/kshape/feature_meta"
NULL_SUMMARY_CSV = os.path.join(LOCAL_META_DIR, "feature_null_summary.csv")
META_JSON = os.path.join(LOCAL_META_DIR, "feature_metadata.json")

BINS_PER_DAY = 48
BINS_PER_WEEK = 7 * BINS_PER_DAY
MISSING_CLUSTER_ID = -1

os.makedirs(LOCAL_META_DIR, exist_ok=True)

spark = SparkSession.builder.appName("FeatureEngineeringIndependent").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.mergeSchema", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

# -------------------------
# 2) HELPERS
# -------------------------
def hdfs_path_exists(path_str: str) -> bool:
    jvm = spark._jvm
    hconf = spark._jsc.hadoopConfiguration()
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(hconf)
    Path = jvm.org.apache.hadoop.fs.Path
    return fs.exists(Path(path_str))

def z_norm_1d(x):
    x = np.asarray(x, dtype=np.float64)
    m = np.nanmean(x)
    s = np.nanstd(x)
    if np.isnan(s) or s < 1e-12:
        s = 1.0
    return (x - m) / s

def load_panel_with_cluster():
    if hdfs_path_exists(PANEL_WITH_CLUSTER_PATH):
        return spark.read.parquet(PANEL_WITH_CLUSTER_PATH), PANEL_WITH_CLUSTER_PATH

    if not hdfs_path_exists(FULL_PANEL_PATH):
        raise ValueError("Không tìm thấy panel_with_cluster_id và cũng không có full_panel fallback.")
    if not os.path.exists(ASSIGNMENTS_CSV):
        raise ValueError("Không tìm thấy panel_with_cluster_id và cũng không có cluster_assignments_final.csv fallback.")

    panel = spark.read.parquet(FULL_PANEL_PATH)
    assignments_pd = pd.read_csv(ASSIGNMENTS_CSV)
    required_cols = {"PULocationID", "cluster_id"}
    missing = required_cols - set(assignments_pd.columns)
    if missing:
        raise ValueError(f"CSV mapping thiếu cột: {sorted(list(missing))}")

    assignments_pd = assignments_pd[["PULocationID", "cluster_id"]].copy()
    assignments_pd["PULocationID"] = pd.to_numeric(assignments_pd["PULocationID"], errors="coerce")
    assignments_pd["cluster_id"] = pd.to_numeric(assignments_pd["cluster_id"], errors="coerce")
    assignments_pd = assignments_pd.dropna(subset=["PULocationID", "cluster_id"]).copy()
    assignments_pd["PULocationID"] = assignments_pd["PULocationID"].astype("int64")
    assignments_pd["cluster_id"] = assignments_pd["cluster_id"].astype("int64")

    assignments_spark = spark.createDataFrame(assignments_pd)
    panel = panel.join(F.broadcast(assignments_spark), on="PULocationID", how="left")
    return panel, f"{FULL_PANEL_PATH} + {ASSIGNMENTS_CSV}"

# -------------------------
# 3) READ PANEL
# -------------------------
panel, input_source = load_panel_with_cluster()

required_core = {"pickup_bin_30m", "PULocationID", "pickup_demand", "dataset_split"}
missing_core = required_core - set(panel.columns)
if missing_core:
    raise ValueError(f"Thiếu cột lõi trong input panel: {sorted(list(missing_core))}")

if "cluster_id" not in panel.columns:
    panel = panel.withColumn("cluster_id", F.lit(MISSING_CLUSTER_ID))
else:
    panel = panel.withColumn("cluster_id", F.coalesce(F.col("cluster_id").cast("int"), F.lit(MISSING_CLUSTER_ID)))

panel = (
    panel
    .withColumn("pickup_bin_30m", F.col("pickup_bin_30m").cast("timestamp"))
    .withColumn("PULocationID", F.col("PULocationID").cast("int"))
    .withColumn("pickup_demand", F.coalesce(F.col("pickup_demand"), F.lit(0.0)).cast("double"))
    .filter(F.col("pickup_bin_30m").isNotNull())
    .filter(F.col("PULocationID").isNotNull())
)

# unify names for downstream
if "slot_30m" in panel.columns and "half_hour_slot" not in panel.columns:
    panel = panel.withColumn("half_hour_slot", F.col("slot_30m").cast("int"))
if "year" not in panel.columns:
    panel = panel.withColumn("year", F.year("pickup_bin_30m").cast("int"))

# only keep rows that were model-ready upstream
if "model_ready" in panel.columns:
    panel = panel.filter(F.col("model_ready") == 1)

panel = panel.persist(StorageLevel.MEMORY_AND_DISK)

print("=" * 120)
print("STEP 4 — FEATURE ENGINEERING")
print("=" * 120)
print(f"INPUT_SOURCE     : {input_source}")
print(f"FEATURE_OUT_PATH : {FEATURE_OUT_PATH}")

# -------------------------
# 4) BUILD CLUSTER TIME SERIES (NO LEAKAGE)
# -------------------------
cluster_ts = (
    panel.filter(F.col("cluster_id") != MISSING_CLUSTER_ID)
         .groupBy("cluster_id", "pickup_bin_30m")
         .agg(F.avg("pickup_demand").alias("cluster_avg_demand_t"))
         .withColumn("year", F.year("pickup_bin_30m").cast("int"))
         .withColumn("month", F.month("pickup_bin_30m").cast("int"))
)

w_cluster = Window.partitionBy("cluster_id").orderBy("pickup_bin_30m")
w_cluster_24h = w_cluster.rowsBetween(-BINS_PER_DAY, -1)

cluster_ts = (
    cluster_ts
    .withColumn("cluster_demand_t_1", F.lag("cluster_avg_demand_t", 1).over(w_cluster))
    .withColumn("cluster_demand_t_2", F.lag("cluster_avg_demand_t", 2).over(w_cluster))
    .withColumn("cluster_demand_t_3", F.lag("cluster_avg_demand_t", 3).over(w_cluster))
    .withColumn("cluster_demand_t_4", F.lag("cluster_avg_demand_t", 4).over(w_cluster))
    .withColumn("cluster_demand_t_5", F.lag("cluster_avg_demand_t", 5).over(w_cluster))
    .withColumn("cluster_rolling_obs_24h", F.count("cluster_avg_demand_t").over(w_cluster_24h))
    .withColumn("cluster_rolling_max_24h", F.max("cluster_avg_demand_t").over(w_cluster_24h))
    .withColumn("cluster_rolling_min_24h", F.min("cluster_avg_demand_t").over(w_cluster_24h))
    .withColumn("cluster_rolling_mean_24h", F.avg("cluster_avg_demand_t").over(w_cluster_24h))
    .withColumn("cluster_rolling_std_24h", F.stddev("cluster_avg_demand_t").over(w_cluster_24h))
)

cluster_ts = cluster_ts.fillna({
    "cluster_demand_t_1": 0.0,
    "cluster_demand_t_2": 0.0,
    "cluster_demand_t_3": 0.0,
    "cluster_demand_t_4": 0.0,
    "cluster_demand_t_5": 0.0,
    "cluster_rolling_obs_24h": 0,
    "cluster_rolling_max_24h": 0.0,
    "cluster_rolling_min_24h": 0.0,
    "cluster_rolling_mean_24h": 0.0,
    "cluster_rolling_std_24h": 0.0,
})

panel = (
    panel
    .join(
        F.broadcast(
            cluster_ts.select(
                "cluster_id", "pickup_bin_30m",
                "cluster_demand_t_1", "cluster_demand_t_2", "cluster_demand_t_3", "cluster_demand_t_4", "cluster_demand_t_5",
                "cluster_rolling_obs_24h", "cluster_rolling_max_24h", "cluster_rolling_min_24h",
                "cluster_rolling_mean_24h", "cluster_rolling_std_24h"
            )
        ),
        on=["cluster_id", "pickup_bin_30m"],
        how="left"
    )
)

panel = panel.fillna({
    "cluster_demand_t_1": 0.0,
    "cluster_demand_t_2": 0.0,
    "cluster_demand_t_3": 0.0,
    "cluster_demand_t_4": 0.0,
    "cluster_demand_t_5": 0.0,
    "cluster_rolling_obs_24h": 0,
    "cluster_rolling_max_24h": 0.0,
    "cluster_rolling_min_24h": 0.0,
    "cluster_rolling_mean_24h": 0.0,
    "cluster_rolling_std_24h": 0.0,
})

# -------------------------
# 5) DERIVED CLUSTER FEATURES MATCHING MODEL INPUT
# -------------------------
panel = panel.withColumn("cluster_Diff_t_1", (F.coalesce(F.col("demand_t_1"), F.lit(0.0)) - F.col("cluster_demand_t_1")).cast("double"))
panel = panel.withColumn("cluster_Mean_diff_24h", (F.coalesce(F.col("rolling_mean_24h"), F.lit(0.0)) - F.col("cluster_rolling_mean_24h")).cast("double"))
panel = panel.withColumn("cluster_Std_diff_24h", (F.coalesce(F.col("rolling_std_24h"), F.lit(0.0)) - F.col("cluster_rolling_std_24h")).cast("double"))
panel = panel.withColumn(
    "has_valid_cluster_feature",
    ((F.col("cluster_id") != MISSING_CLUSTER_ID) & (F.col("cluster_demand_t_1").isNotNull())).cast("int")
)

# -------------------------
# 6) STATIC SIMILARITY (TRAIN ONLY)
# -------------------------
panel_train = panel.filter(F.col("dataset_split") == "train")

series_df = (
    panel_train.select("PULocationID", "cluster_id", "pickup_bin_30m", "pickup_demand")
              .orderBy("PULocationID", "pickup_bin_30m")
              .groupBy("PULocationID", "cluster_id")
              .agg(F.collect_list(F.struct("pickup_bin_30m", "pickup_demand")).alias("pairs"))
              .select(
                  "PULocationID", "cluster_id",
                  F.expr("transform(pairs, x -> cast(x.pickup_demand as double))").alias("series")
              )
              .orderBy("PULocationID")
)

series_rows = series_df.collect()

if len(series_rows) > 0:
    loc_ids = [int(r["PULocationID"]) for r in series_rows]
    loc_clusters = [int(r["cluster_id"]) for r in series_rows]
    X = np.array([r["series"] for r in series_rows], dtype=np.float64)
    Xz = np.vstack([z_norm_1d(x) for x in X])

    cluster_ids_unique = sorted(set(loc_clusters))
    cluster_reps = {}
    for cid in cluster_ids_unique:
        idx = [i for i, c in enumerate(loc_clusters) if c == cid]
        rep = Xz[idx].mean(axis=0)
        rep = z_norm_1d(rep)
        cluster_reps[cid] = rep

    sim_rows = []
    for i, loc_id in enumerate(loc_ids):
        cid = loc_clusters[i]
        x = Xz[i]
        own_rep = cluster_reps[cid]
        intra_sim = float(np.mean(x * own_rep))

        other_sims = []
        for other_cid, other_rep in cluster_reps.items():
            if other_cid == cid:
                continue
            other_sims.append(float(np.mean(x * other_rep)))
        inter_sim = max(other_sims) if len(other_sims) > 0 else None

        sim_rows.append({
            "PULocationID": int(loc_id),
            "intra_cluster_similarity": intra_sim,
            "inter_cluster_similarity": inter_sim
        })

    sim_pd = pd.DataFrame(sim_rows)
    sim_spark = spark.createDataFrame(sim_pd)
    panel = panel.join(sim_spark, on="PULocationID", how="left")
else:
    panel = panel.withColumn("intra_cluster_similarity", F.lit(None).cast("double"))
    panel = panel.withColumn("inter_cluster_similarity", F.lit(None).cast("double"))

panel = panel.fillna({
    "intra_cluster_similarity": 0.0,
    "inter_cluster_similarity": 0.0
})

# -------------------------
# 7) FINAL COLUMN SELECTION
# -------------------------
candidate_cols = [
    "pickup_bin_30m", "date", "pickup_date", "dataset_split", "PULocationID",
    "pickup_demand",
    "hour", "minute", "slot_30m", "half_hour_slot", "day_of_week", "week_of_year", "month", "year", "day_of_month",
    "is_weekday", "is_weekend",
    "demand_t_1", "demand_t_2", "demand_t_3", "demand_t_4", "demand_t_5",
    "rolling_obs_24h", "rolling_max_24h", "rolling_min_24h", "rolling_mean_24h", "rolling_std_24h", "rolling_skew_24h",
    "ha_output", "ewma_output",
    "history_bins_available", "enough_history_lag5", "enough_history_24h", "enough_history_ewma", "enough_history_1w",
    "is_cold_start", "model_ready",
    "cluster_id",
    "cluster_demand_t_1", "cluster_demand_t_2", "cluster_demand_t_3", "cluster_demand_t_4", "cluster_demand_t_5",
    "cluster_rolling_obs_24h", "cluster_rolling_max_24h", "cluster_rolling_min_24h", "cluster_rolling_mean_24h", "cluster_rolling_std_24h",
    "cluster_Diff_t_1", "cluster_Mean_diff_24h", "cluster_Std_diff_24h",
    "intra_cluster_similarity", "inter_cluster_similarity",
    "has_valid_cluster_feature"
]

final_cols = [c for c in candidate_cols if c in panel.columns]
df_final = panel.select(*[F.col(c) for c in final_cols]).persist(StorageLevel.DISK_ONLY)

# -------------------------
# 8) NULL SUMMARY
# -------------------------
null_counts_row = df_final.agg(*[
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in final_cols
]).first().asDict()
total_rows = df_final.count()

null_summary_pd = pd.DataFrame({
    "column": list(null_counts_row.keys()),
    "null_count": list(null_counts_row.values())
})
null_summary_pd["null_ratio"] = null_summary_pd["null_count"] / max(total_rows, 1)
null_summary_pd = null_summary_pd.sort_values(["null_ratio", "column"], ascending=[False, True]).reset_index(drop=True)
null_summary_pd.to_csv(NULL_SUMMARY_CSV, index=False)

# -------------------------
# 9) SAVE OUTPUT
# -------------------------
partition_cols = []
if "dataset_split" in df_final.columns:
    partition_cols.append("dataset_split")
if "year" in df_final.columns:
    partition_cols.append("year")
if "month" in df_final.columns:
    partition_cols.append("month")

writer = df_final.write.mode("overwrite")
if partition_cols:
    writer = writer.partitionBy(*partition_cols)
writer.parquet(FEATURE_OUT_PATH)

meta = {
    "input_source": input_source,
    "panel_with_cluster_path": PANEL_WITH_CLUSTER_PATH,
    "fallback_full_panel_path": FULL_PANEL_PATH,
    "fallback_assignments_csv": ASSIGNMENTS_CSV,
    "feature_out_path": FEATURE_OUT_PATH,
    "null_summary_csv": NULL_SUMMARY_CSV,
    "row_count": int(total_rows),
    "feature_columns": final_cols,
    "notes": [
        "cluster_avg_demand_t hiện tại KHÔNG được join trực tiếp để tránh leakage.",
        "cluster_Diff_t_1 = demand_t_1 - cluster_demand_t_1.",
        "cluster_Mean_diff_24h = rolling_mean_24h - cluster_rolling_mean_24h.",
        "cluster_Std_diff_24h = rolling_std_24h - cluster_rolling_std_24h.",
        "static similarity được tính trên train split."
    ]
}
with open(META_JSON, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 120)
print("STEP 4 DONE")
print("=" * 120)
print(f"Feature rows   : {total_rows}")
print(f"Feature cols   : {len(final_cols)}")
print(f"Feature out    : {FEATURE_OUT_PATH}")
print(f"Null summary   : {NULL_SUMMARY_CSV}")
print(f"Metadata json  : {META_JSON}")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-df8a44d3-9f8c-4870-bfb1-84c272e9aaa3;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 120ms :: artifacts dl 5ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

STEP 4 — FEATURE ENGINEERING
INPUT_SOURCE     : /user/data/clustering/panel_with_cluster_id
FEATURE_OUT_PATH : /user/data/feature_engineering/feature_panel_model_ready



STEP 4 DONE
Feature rows   : 456060
Feature cols   : 53
Feature out    : /user/data/feature_engineering/feature_panel_model_ready
Null summary   : /workspace/code/kshape/feature_meta/feature_null_summary.csv
Metadata json  : /workspace/code/kshape/feature_meta/feature_metadata.json
